# HuGo Intelligence 2.0 — Vietnamese Sentiment TF-IDF MVP

Notebook này minh họa pipeline phân loại sentiment tiếng Việt ba lớp: `NEGATIVE`, `NEUTRAL`, `POSITIVE`. Nó tái sử dụng trực tiếp `02_vietnamese_sentiment_tfidf_mvp.py`; không viết lại thuật toán.

> **DEMO_ONLY — Minh họa pipeline, không phải đánh giá chất lượng mô hình.**
>
> `data_mode=DEMO_ONLY` · `quality_claim_allowed=false` · `production_eligible=false`

45 câu trong fixture do người thực hiện tự viết để kiểm tra kỹ thuật. Chúng không phải review người dùng, development dataset hoặc independent test. Metric hiển thị bên dưới không được dùng để khẳng định chất lượng AI và model không được activate cho website.

## 1. Chuẩn bị Kaggle — CPU, Internet Off

Notebook chỉ cần hai file đầu vào, đặt trong một Kaggle Dataset **Private**:

- `02_vietnamese_sentiment_tfidf_mvp.py`
- `vietnamese_sentiment_demo.jsonl`

Có thể giữ cấu trúc `ai/kaggle/` và `ai/fixtures/`, hoặc upload hai file ở thư mục gốc. Cell kế tiếp tự tìm file theo tên và từ chối nếu tìm thấy nhiều bản.

Trong **Notebook options / Session options** chọn:

- Accelerator: `None`
- Internet: `Off`
- Notebook/Dataset visibility: `Private`

Không attach database dump, HuGo test/production export hoặc dataset bên ngoài.

In [ ]:
from pathlib import Path
from collections import Counter
import argparse
import importlib.util
import json
import os
import sys
import tempfile

from IPython.display import Image, Markdown, display
import joblib
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "hugo_m2_b2_matplotlib"))
import matplotlib.pyplot as plt
import numpy as np

DISCLAIMER = "Minh họa pipeline — không phải đánh giá chất lượng mô hình"
EXPECTED_GOVERNANCE = {
    "data_mode": "DEMO_ONLY",
    "quality_claim_allowed": False,
    "production_eligible": False,
}

def find_required_file(relative_path):
    relative_path = Path(relative_path)
    current = Path.cwd().resolve()
    for root in [current, *current.parents]:
        candidate = root / relative_path
        if candidate.is_file():
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    matches = list(kaggle_input.rglob(relative_path.name)) if kaggle_input.is_dir() else []
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Cần đúng 1 file {relative_path.name}; tìm thấy {len(matches)} tại /kaggle/input"
        )
    return matches[0].resolve()

PIPELINE_PATH = find_required_file("ai/kaggle/02_vietnamese_sentiment_tfidf_mvp.py")
FIXTURE_PATH = find_required_file("ai/fixtures/vietnamese_sentiment_demo.jsonl")
IS_KAGGLE = Path("/kaggle/working").is_dir()
OUTPUT_DIR = (
    Path("/kaggle/working/hugo_vi_sentiment_m2_b2_demo")
    if IS_KAGGLE
    else PIPELINE_PATH.parents[1] / "artifacts" / "m2_b2_notebook_local"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Execution environment:", "KAGGLE_CPU" if IS_KAGGLE else "LOCAL_CPU")
print("Pipeline:", PIPELINE_PATH)
print("Fixture:", FIXTURE_PATH)
print("Output:", OUTPUT_DIR)
print(json.dumps(EXPECTED_GOVERNANCE, ensure_ascii=False))
print(DISCLAIMER)

## 2. Đọc và kiểm tra 45 câu DEMO_ONLY

Cell này import module M2-B1 từ file đã attach rồi gọi lại chính các hàm đọc, NFC normalization và validation. Không có download hoặc kết nối database.

In [ ]:
spec = importlib.util.spec_from_file_location("hugo_m2_b1_pipeline", PIPELINE_PATH)
if spec is None or spec.loader is None:
    raise RuntimeError("Không thể import pipeline M2-B1")
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

raw_records = pipeline.read_records(FIXTURE_PATH)
records, fixture_audit = pipeline.clean_and_validate_records(raw_records, "DEMO_ONLY")

assert len(records) == 45, f"Expected 45 demo rows, got {len(records)}"
assert all(row["data_tier"] == "DEMO_ONLY" for row in records)
assert all(row["quality_claim_allowed"] is False for row in records)
assert all(row["production_eligible"] is False for row in records)
assert fixture_audit["unicode_normalization"] == "NFC"

print(json.dumps(fixture_audit, ensure_ascii=False, indent=2))
print("Ba dòng đầu (dữ liệu minh họa do người thực hiện tự viết):")
for row in records[:3]:
    print({key: row[key] for key in ("id", "text", "label", "domain")})

In [ ]:
label_counts = Counter(row["label"] for row in records)
domain_counts = Counter(row["domain"] for row in records)
assert [label_counts[label] for label in pipeline.LABELS] == [15, 15, 15]
assert domain_counts == {"hotel": 15, "restaurant": 15, "attraction": 15}

figure, axis = plt.subplots(figsize=(7, 4))
values = [label_counts[label] for label in pipeline.LABELS]
bars = axis.bar(pipeline.LABELS, values, color=["#d95f5f", "#8a8a8a", "#56a66f"])
axis.bar_label(bars)
axis.set_ylim(0, 18)
axis.set_ylabel("Số câu minh họa")
axis.set_title("Phân bố nhãn — DEMO_ONLY")
figure.text(0.5, 0.01, DISCLAIMER, ha="center", color="darkred", fontweight="bold")
figure.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()
print("Label distribution:", dict(label_counts))
print("Domain distribution:", dict(domain_counts))

## 3. Chạy bốn baseline từ pipeline M2-B1

`run_pipeline()` chạy majority baseline, word TF-IDF 1–2, `char_wb` TF-IDF 3–5 và combined word+character với Logistic Regression. Vectorizer chỉ fit trên train split.

Các metric chỉ giúp xác nhận code chạy và artifact được tạo. Chúng không phải benchmark chất lượng.

In [ ]:
arguments = argparse.Namespace(
    development_data=str(FIXTURE_PATH),
    independent_test_data=None,
    approved_manifest=None,
    output_dir=str(OUTPUT_DIR),
)
run_result = pipeline.run_pipeline(arguments)

for key, expected in EXPECTED_GOVERNANCE.items():
    assert run_result["governance"][key] == expected
assert run_result["reload_parity"] == "PASS"
print("TECHNICAL PIPELINE RUN: PASS")
print(DISCLAIMER)

In [ ]:
with (OUTPUT_DIR / "metrics_demo_only.json").open(encoding="utf-8") as handle:
    metrics = json.load(handle)

for key, expected in EXPECTED_GOVERNANCE.items():
    assert metrics["governance"][key] == expected

display(Markdown("### Metric kỹ thuật — không phải kết quả chất lượng AI"))
print(f"{'Model':52} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 74)
for name, values in metrics["models"].items():
    print(f"{name:52} {values['accuracy']:10.4f} {values['macro_f1']:10.4f}")
print()
print("Evaluation tier:", metrics["models"]["combined_word_char_tfidf_logistic_regression"]["evaluation_tier"])
print(DISCLAIMER)

## 4. Confusion matrix có watermark DEMO_ONLY

Ma trận dưới đây có 5 câu/lớp trong demo holdout. Không diễn giải nó như chất lượng trên review thực tế.

In [ ]:
confusion_path = OUTPUT_DIR / "confusion_matrix_demo_only.png"
assert confusion_path.is_file()
display(Image(filename=str(confusion_path)))
print("Watermark:", DISCLAIMER)

## 5. Artifact, SHA-256 và reload parity

Chỉ load joblib do chính run này tạo. Joblib/pickle từ nguồn không tin cậy có thể thực thi code.

In [ ]:
with (OUTPUT_DIR / "artifact_checksums.json").open(encoding="utf-8") as handle:
    checksum_manifest = json.load(handle)
with (OUTPUT_DIR / "reload_parity.json").open(encoding="utf-8") as handle:
    reload_report = json.load(handle)

checksum_results = {}
for name, expected in checksum_manifest["files"].items():
    actual = pipeline.sha256_file(OUTPUT_DIR / name)
    checksum_results[name] = actual == expected["sha256"]
    print(f"{name:45} {'PASS' if checksum_results[name] else 'FAIL'}  {actual}")

assert all(checksum_results.values())
assert reload_report["status"] == "PASS"
assert reload_report["predicted_labels_equal"] is True
assert reload_report["predicted_probabilities_equal"] is True
for key, expected in EXPECTED_GOVERNANCE.items():
    assert checksum_manifest["governance"][key] == expected
    assert reload_report["governance"][key] == expected

print("CHECKSUM VERIFICATION: PASS")
print("RELOAD PARITY: PASS")
print(DISCLAIMER)

## 6. Inference thử với câu do bạn tự nhập

Sửa danh sách `USER_SENTENCES` rồi chạy lại riêng cell dưới. Không nhập tên, email, số điện thoại, booking/account ID hoặc nội dung của người khác.

Prediction chỉ minh họa giao diện kỹ thuật. Model học từ 30 câu demo nên kết quả có thể sai và không được dùng cho quyết định thực tế.

In [ ]:
# Hãy thay các câu dưới đây bằng câu tiếng Việt do chính bạn nhập.
USER_SENTENCES = [
    "Phòng rất sạch và nhân viên hỗ trợ nhanh.",
    "Khách sạn có một thang máy và hai lối vào.",
    "Món ăn không ngon và phục vụ quá chậm.",
]

sentences = [pipeline.normalize_text(value) for value in USER_SENTENCES]
sentences = [value for value in sentences if value]
if not sentences:
    raise ValueError("Hãy nhập ít nhất một câu không rỗng")

artifact_path = OUTPUT_DIR / "combined_pipeline_demo_only.joblib"
payload = joblib.load(artifact_path)
for key, expected in EXPECTED_GOVERNANCE.items():
    assert payload["governance"][key] == expected

model = payload["pipeline"]
predictions = model.predict(sentences)
probabilities = model.predict_proba(sentences)
classes = list(model.named_steps["classifier"].classes_)

for sentence, prediction, probability in zip(sentences, predictions, probabilities):
    scores = {label: round(float(score), 4) for label, score in zip(classes, probability)}
    print({"text": sentence, "demo_prediction": prediction, "demo_scores": scores})

print(DISCLAIMER)
print("Không dùng prediction này để đánh giá người dùng, địa điểm hoặc chất lượng dịch vụ.")

In [ ]:
required_artifacts = {
    "metrics": OUTPUT_DIR / "metrics_demo_only.json",
    "confusion_png": OUTPUT_DIR / "confusion_matrix_demo_only.png",
    "pipeline": OUTPUT_DIR / "combined_pipeline_demo_only.joblib",
    "metadata": OUTPUT_DIR / "model_metadata.json",
    "checksums": OUTPUT_DIR / "artifact_checksums.json",
    "reload_parity": OUTPUT_DIR / "reload_parity.json",
}
final_checks = {
    "fixture_has_45_rows": len(records) == 45,
    "three_balanced_labels": [label_counts[label] for label in pipeline.LABELS] == [15, 15, 15],
    "all_artifacts_exist": all(path.is_file() for path in required_artifacts.values()),
    "all_checksums_match": all(checksum_results.values()),
    "reload_parity_pass": reload_report["status"] == "PASS",
    "demo_only": run_result["governance"]["data_mode"] == "DEMO_ONLY",
    "quality_claim_disabled": run_result["governance"]["quality_claim_allowed"] is False,
    "production_disabled": run_result["governance"]["production_eligible"] is False,
}
print(json.dumps(final_checks, ensure_ascii=False, indent=2))
assert all(final_checks.values())
print("M2-B2 TECHNICAL DEMO: PASS in", "KAGGLE_CPU" if IS_KAGGLE else "LOCAL_CPU")
print("KAGGLE EXECUTION: RUN BY USER" if IS_KAGGLE else "KAGGLE NOT RUN")
print(DISCLAIMER)

## Hoàn tất demo kỹ thuật

Nếu mọi cell PASS, pipeline có thể tái lập trong môi trường đang chạy. Điều này **không** thay đổi M2 data gate, không chứng minh model đủ chất lượng và không cho phép activate model.

Artifact nằm trong `/kaggle/working/hugo_vi_sentiment_m2_b2_demo` khi chạy trên Kaggle. Chỉ tải về local sau khi kiểm checksum; không publish notebook/dataset/model.